In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Preprocessing 
* Pre merge EDA
* Data merge
* Post merge EDA

In [ ]:
!git pull

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import random
import string


from sklearn.model_selection import train_test_split

In [ ]:
# %cd C:/Users/arpit/spring-2026-deep-learning-fragmented-id-resolution
%cd ~/spring-2026-deep-learning-fragmented-id-resolution

In [ ]:
!git checkout Pedro

In [ ]:
# Pull ncvoters.tsv from Pedro branch without overwriting
import subprocess
import os

# Extract the file from Pedro branch and save it locally
subprocess.run([
    'git', 'show', 'Pedro:data/raw/ncvoters.tsv'
], stdout=#open('data/raw', 'w'), cwd='.')
open('data/raw/ncvoters_from_Pedro.tsv', 'w'))
print("✓ Extracted ncvoters.tsv from Pedro branch → data/raw/ncvoters_from_Pedro.tsv")

# Read the TSV files into the DataFrame
# Data Source Hasso Plattner Institut 
# NDPL - Non-duplicates
# DPL - Duplicates
# ncvoters - a snap shot of the snapshot: VR_Snapshot_20181106 
df_ncvoters = pd.read_csv(r'data/raw/ncvoters.tsv', sep='\t')
DPL = pd.read_csv(r'data/raw/ncvoters_DPL.tsv', sep='\t')
NDPL = pd.read_csv(r'data/raw/ncvoters_NDPL.tsv', sep='\t')

print(f"✓ Loaded ncvoters: {df_ncvoters.shape}")
print(f"✓ Loaded DPL: {DPL.shape}")
print(f"✓ Loaded NDPL: {NDPL.shape}")


In [ ]:
# Data distribution
counts = [14183, 9819, 98142]
labels = ["Records", "Duplicate pairs", "Non-duplicate pairs"]

plt.bar(labels, counts)
plt.title("Dataset Composition")
plt.show()

No duplicates 

In [ ]:
print(df_ncvoters['id'].duplicated().any())



In [ ]:
df_ncvoters.head()

In [ ]:
# Print the first few rows of the DataFrame
print(df_ncvoters.head())

In [ ]:
print(DPL.head())
print(NDPL.head())

In [ ]:
NDPL.info()

# Identification

Testing to ensure intersection of ids across data sets



The Hasso Plattner Institute created a unique id in snapshot: VR_Snapshot_20181106, in this data they use "id" as the identification in the ncvoters data ( not ncid or voter registration number).
The next few code boxes show an intersection between all the id variables across the data set. 

In [ ]:
# #Quick check for intersection across ids from the different tables
print(set(DPL["id1"]).intersection(set(df_ncvoters["id"])))
print(set(NDPL["id1"]).intersection(set(df_ncvoters["id"])))
print(set(NDPL["id1"]).intersection(set(DPL["id1"])))
print(set(DPL["id2"]).intersection(set(df_ncvoters["id"])))
print(set(NDPL["id2"]).intersection(set(df_ncvoters["id"])))

In [ ]:
print(NDPL["id1"].head())
print(NDPL["id1"].tail())
print(DPL["id2"].head())
print(DPL["id2"].tail())
print(df_ncvoters["id"].head())
print(df_ncvoters["id"].tail())


In [ ]:
# Top 20 names 

# 1. Create a combined 'full_name' column
#  Use .fillna('') to ensure missing names don't break the combination
df_ncvoters['full_name'] = df_ncvoters['first_name'].fillna('').str.strip() + " " + df_ncvoters['last_name'].fillna('').str.strip()

# 2. Use value_counts() to find the most common combinations
top_20_names = df_ncvoters['full_name'].value_counts().head(20)

# 3. Display the result
print("Top 20 Most Common First + Last Names in County:")
print(top_20_names)

In [ ]:
# Checking for missing data
df_ncvoters.isna().mean().sort_values(ascending=False)

In [ ]:
missing = df_ncvoters.isna().mean().sort_values(ascending=True)

missing[missing > 0].plot.barh(figsize=(6,8))
plt.title("Field Missingness")
plt.xlabel("Proportion Missing")
plt.show()

Uniqueness and entropy -tells us how "useful" are feature is for distinguishing one person from another.

Uniqueness/cardinality of fields:

* Street name has the strongest
* Name suffix is the weakest (however we should keep suffix)

In [ ]:
# 1. Define  columns
cols = ["full_name", "first_name","midl_name", "last_name", "name_sufx_cd", "age", "age_group", "race_code", "ethnic_code", "sex", "house_num", 
        "street_name", "street_type_cd", "unit_num", "zip_code", "res_city_desc", "area_cd", "phone_num"]

# 2. Compute nunique and ratio simultaneously
summary = df_ncvoters[cols].agg(['nunique', lambda x: x.nunique() / len(x)]).T

# 3. Rename columns and sort
summary.columns = ['unique_count', 'ratio']
summary = summary.sort_values(by='ratio', ascending=False)

print(summary)


In [ ]:
full = df_ncvoters["first_name"] + " " + df_ncvoters["last_name"]

full.value_counts().hist(bins=50)
plt.title("Full Name Frequency Distribution")
plt.xlabel("Occurrences")
plt.show()

Entropy - measure of the "uncertainty" or "disorder" of the data distribution.
How evenly is the information distributed. High entropy is better for probabilistic matching. Even if uniqueness is low, if the data is distributed evenly, it provides more "discerning power" to our algorithm.


"For an identity detection project, the ideal column should have high entropy indicating that the data is highly unique, random, and not repetitive. A high entropy value means the column provides maximum information for distinguishing between entities, while low entropy implies redundant or predictable data. 

Key Considerations for Identity Columns:
High Entropy = Better Identity Detection: Columns like User IDs, UUIDs, email addresses, or biometric hashes should have high entropy because each value is unique and unpredictable.

Low Entropy = Poor Identifier: Columns with repetitive data (e.g."Gender" or "City" in a small dataset) have low entropy, making them unsuitable as primary, unique identifiers."

In [ ]:
cols = ["full_name", "first_name","midl_name", "last_name", "name_sufx_cd", "age", "age_group", "race_code", "ethnic_code", "sex", "house_num", 
        "street_name", "street_type_cd", "unit_num", "zip_code", "res_city_desc", "area_cd", "phone_num"]
results = []

for col in cols:
    p = df_ncvoters[col].value_counts(normalize=True)
    entropy = -(p * np.log2(p)).sum()
    results.append((col, entropy))

# sort by entropy (descending)
results.sort(key=lambda x: x[1], reverse=True)

for col, ent in results:
    print(f"{col} entropy: {ent:.4f}")

In [ ]:
df_ncvoters["full_name"].str.len().hist(bins=20)
plt.title("Full Name Length Distribution")
plt.show()

In [ ]:
df_ncvoters["first_name"].str.len().hist(bins=20)
plt.title("First Name Length Distribution")
plt.show()

In [ ]:
df_ncvoters["last_name"].str.len().hist(bins=20)
plt.title("Last Name Length Distribution")
plt.show()

In [ ]:
df_ncvoters["street_name"].str.len().hist(bins=20)
plt.title("Street Name Length Distribution")
plt.show()

In [ ]:
df_ncvoters["house_num"].hist(bins=20)
plt.title("House Number Length Distribution")
plt.show()

In [ ]:
cols = ["street_name", "last_name", "first_name", "zip_code", "mail_zipcode", "age"]

uniq = [df_ncvoters[c].nunique() / len(df_ncvoters) for c in cols]

plt.bar(cols, uniq)
plt.title("Field Uniqueness Ratio")
plt.ylabel("Unique / Total")
plt.show()

In [ ]:
cols = ["full_name", "first_name","midl_name", "last_name", "name_sufx_cd", "age", "age_group", "race_code", "ethnic_code", "sex", "house_num", 
        "street_name", "street_type_cd", "unit_num", "zip_code", "res_city_desc", "area_cd", "phone_num"]
results = []

In [ ]:
# Preprocessing for our fragemented ID analysis 
# Selecting identifyer variables not related to voting
df_ncvoters_frag_ID = df_ncvoters[[
#Stable Identifiers
#Only to be used for labeling /evaluation only 
# / not as a model feature
    'id', 'ncid', 'voter_reg_num', 
# Primary Model features - Strongest features, Strong entropy, Essential for matching
    #many variable such as name prefx and sufx are sparse we can either use none for missing or 0/1
    'full_name', 'first_name', 'midl_name', 'last_name', 'name_sufx_cd',
    #other varaiabes
# Adress Similarity features - Address is the second strongest identity anchor, 
    # Street name especially high discriminative signal # Unit numbers distinguish household
    #many variable such as unit designator are sparse we can either use none for missing or 0/1
    'house_num', 'street_name', 'street_dir', 'street_type_cd', 'unit_designator', 'unit_num', 'zip_code', 'res_city_desc',
#other varaiabes
# Demographic agreement indicators - Moderate/Supporting Features 
    # these will help us reduce false matches # they are agreement indicators, low-weight similarity features
    #age group rather than age because grouped/bin age is more stable
    'age', 'age_group', 'sex', 'race_code', 'race_desc', 'ethnic_code', 'ethnic_desc', 'birth_place',
# #other varaiabes 
 'phone_num','area_cd'   
 ]]
# print("\nSelected variables 'A' and 'C':")
print(df_ncvoters_frag_ID)

~5.27% of suffixes are present, 94.73% are missing. Representative of real world population. Important for seperating father/son at the same address, prevents false merges. This is strong for entity resolution. 

In [ ]:
# 1. Get the raw counts
counts = df_ncvoters_frag_ID['name_sufx_cd'].value_counts(dropna=False)

# 2. Get the percentages (normalize=True) and multiply by 100
percent = df_ncvoters_frag_ID['name_sufx_cd'].value_counts(dropna=False, normalize=True) * 100

# 3. Combine them into a table
summary = pd.concat([counts, percent], axis=1)
summary.columns = ['Count', 'Percentage (%)']

print(summary)


In [ ]:
# Visualize suffix 
plt.figure(figsize=(10, 6))
ax = counts.plot(kind='bar', color='skyblue', edgecolor='black')

# Add the Percentage labels on top of the bars
for i, p in enumerate(percent):
    ax.annotate(f'{p:.1f}%', 
                (i, counts.iloc[i]), 
                ha='center', va='bottom', 
                xytext=(0, 5), textcoords='offset points',
                fontsize=10, fontweight='bold')

plt.title('Suffix Distribution (Count + %)')
plt.ylabel('Number of Records')
plt.xticks(rotation=0) 
plt.show()


# Dealing with missing observations  

The data does not have missing information in terms of nulls/NAN, it is more so empty spaces and these spaces are placeholders/important signal.

In [ ]:
#Summary Statistics 
# Generate summary for ALL variables (numeric and categorical)
summary_all = df_ncvoters_frag_ID.describe(include='all').T

# Add a 'Missing' column to show data quality for each variable
summary_all['missing'] = len(df_ncvoters_frag_ID) - summary_all['count']

# Format the table for a report (rounding decimals)
summary_all = summary_all.round(2)

print("--- Data Overview: All Variables ---")
print(summary_all)


In [ ]:
df_ncvoters_frag_ID.info()

In [ ]:
summary_all.style.background_gradient(cmap='Reds', subset=['missing'])


# Merging data 
Adding labels and merging the three data sets (ncvoter_fragmentedID, DPL and NPL). 
* df_ncvoters_frag_ID - Preprocessed data set with relevant variables.
* df_ncvoters_DPL - A list of all provided duplicates.
* df_ncvoters_NDPL - Non-duplicate pairs

# Next step Creating pairwise features 
combining duplicates and non duplicates

In [ ]:
df_ncvoters_frag_ID.info()

In [ ]:
# Adding labels to DPL and NDPL
DPL['label'] = 1 
NDPL['label'] = 0


print(DPL.head(10))
print(NDPL.head(10))

In [ ]:
pairs = pd.concat([DPL, NDPL])

print(pairs.head(20))
print(pairs.tail(20))

In [ ]:
df_ncvoters_frag_ID.info()

In [ ]:
# merging ncvoters on DPL and NDPL
pairs = pairs.merge(
    df_ncvoters_frag_ID,
    left_on="id1",
    right_on="id",
    how="left"
)
print(pairs.info())
print(pairs.head(10))

In [ ]:
#merging id2 
pairs = pairs.merge(
    df_ncvoters_frag_ID,
    left_on="id2",
    right_on="id",
    how="left",
    suffixes=("_1", "_2")
)

print(pairs.info())
print(pairs.head(10))

## EDA post merges

* Checking to see if duplicates actually look similar. 
  * Class balance 
  * Label separability
  * missingness 


In [ ]:
pairs.head()

In [ ]:
#Confirming duplicates are present # Output: True
print(pairs['id_1'].duplicated().any()

)

In [ ]:
# Class balance 
pairs["label"].value_counts(normalize=True)

Same name rates, the duplicates > non-duplicates -> good seperability 

In [ ]:
#Test
pairs["same_last_name"] = (
    pairs["last_name_1"].str.lower().fillna("") ==
    pairs["last_name_2"].str.lower().fillna("")
).astype(int)


In [ ]:
pairs.groupby("label")["same_last_name"].mean()

In [ ]:
#Test
pairs["age"] = (
    pairs["age_1"] ==
    pairs["age_2"]
).astype(int)

In [ ]:
pairs.groupby("label")["age"].mean()

In [ ]:
pairs["same_address"] = (
    pairs["house_num_1"] ==
    pairs["house_num_2"]
).astype(int)

pairs.groupby("label")["same_address"].mean()

In [ ]:
pairs["same_street_name"] = (
    pairs["street_name_1"].str.lower().fillna("") ==
    pairs["street_name_2"].str.lower().fillna("")
).astype(int)

pairs.groupby("label")["same_street_name"].mean()

In [ ]:
#Overall Separability Check
from rapidfuzz.fuzz import ratio

pairs["first_name_sim"] = pairs.apply(
    lambda x: ratio(
        str(x["first_name_1"]),
        str(x["first_name_2"])
    ),
    axis=1
)

pairs["last_name_sim"] = pairs.apply(
    lambda x: ratio(
        str(x["last_name_1"]),
        str(x["last_name_2"])
    ),
    axis=1
)

In [ ]:
pairs.groupby("label")[["first_name_sim","last_name_sim"]].mean()

In [ ]:
#Overall Separability Check 
from sklearn.metrics import roc_auc_score

features = [
    "same_last_name",
    "same_address",
    "first_name_sim",
    "last_name_sim",
    "age"
]

for f in features:
    clean = pairs[[f, "label"]].dropna()
    auc = roc_auc_score(clean["label"], clean[f])
    print(f"{f}: AUC = {auc:.3f}")

In [ ]:
# sns.kdeplot(data=pairs, x="first_name_sim", hue="label")

In [ ]:
#Hard Negatives - the cases our model might struggle with  
pairs[
    (pairs["label"] == 0) &
    (pairs["last_name_sim"] > 85)
].head(20)

In [ ]:
exact_cols = [
    "first_name",
    "last_name",
    "midl_name",
    "house_num",
    "street_name",
    "zip_code",
    "res_city_desc",
    "sex",
    "race_desc",
    "ethnic_desc"
]

for col in exact_cols:
    pairs[f"{col}_exact"] = (
        pairs[f"{col}_1"] == pairs[f"{col}_2"]
    ).astype(int)

In [ ]:
pairs["age_diff"] = abs(pairs["age_1"] - pairs["age_2"])
pairs["age_exact"] = (pairs["age_diff"] == 0).astype(int)
pairs["age_close"] = (pairs["age_diff"] <= 1).astype(int)
pairs.head(50)

Creating binary columns for names that have the same phonetic pronounciation - Real duplicates often differ by spelling but not pronunciation. Phonetic match = strong evidence of same person.

In [ ]:
import jellyfish

def phonetic_features(pairs, col):

   pairs[f"{col}_soundex_1"] = pairs[f"{col}_1"].fillna("").apply(jellyfish.soundex)
   pairs[f"{col}_soundex_2"] = pairs[f"{col}_2"].fillna("").apply(jellyfish.soundex)

   pairs[f"{col}_metaphone_1"] = pairs[f"{col}_1"].fillna("").apply(jellyfish.metaphone)
   pairs[f"{col}_metaphone_2"] = pairs[f"{col}_2"].fillna("").apply(jellyfish.metaphone)

   pairs[f"{col}_soundex_match"] = (
        pairs[f"{col}_soundex_1"] == pairs[f"{col}_soundex_2"]
    ).astype(int)

   pairs[f"{col}_metaphone_match"] = (
        pairs[f"{col}_metaphone_1"] == pairs[f"{col}_metaphone_2"]
    ).astype(int)

   pairs.head(50)

In [ ]:
for col in ["first_name", "last_name"]:
    phonetic_features(pairs, col)

In [ ]:
#Missing agreement future 
for col in ["midl_name", "phone_num"]:
    pairs[f"{col}_both_missing"] = (
        (pairs[f"{col}_1"] == "") &
        (pairs[f"{col}_2"] == "")
    ).astype(int)

In [ ]:
pairs.groupby("label")[[
    "first_name_soundex_match",
    "last_name_soundex_match"
]].mean()

In [ ]:
pairs.groupby("label")[[
    "first_name_metaphone_match",
    "last_name_metaphone_match"
]].mean()

These are the features we will use to build Siamese network and the features that will be used for modeling, respectively.

In [ ]:
# ========== SIAMESE NETWORK FEATURES ==========
# Raw text fields that will be embedded
siamese_features = [
    # Names
    'first_name_1', 'first_name_2',
    'midl_name_1', 'midl_name_2', 
    'last_name_1', 'last_name_2',
    'name_sufx_cd_1', 'name_sufx_cd_2',
    
    # Address
    'house_num_1', 'house_num_2',
    'street_name_1', 'street_name_2',
    'street_type_cd_1', 'street_type_cd_2',
    'unit_num_1', 'unit_num_2',
    'zip_code_1', 'zip_code_2',
    'res_city_desc_1', 'res_city_desc_2',
    
    # Demographics
    'age_1', 'age_2',
    'phone_num_1', 'phone_num_2',
    'sex_1', 'sex_2',
    'race_desc_1', 'race_desc_2'
]

# ========== BASELINE MODEL FEATURES ==========
# Engineered comparison features for traditional ML
model_features = [
    # Exact matches
    'same_last_name', 'same_street_name',
    'first_name_exact', 'last_name_exact', 'midl_name_exact',
    'house_num_exact', 'street_name_exact', 'zip_code_exact',
    'res_city_desc_exact', 'sex_exact', 'race_desc_exact',
    # Similarity scores
    'first_name_sim', 'last_name_sim', 
    
    # Age comparisons
    'age_diff', 'age_exact', 'age_close',
    
    # Phonetic matches
    'first_name_soundex_match', 'first_name_metaphone_match',
    'last_name_soundex_match', 'last_name_metaphone_match',
    
    # Missingness indicators
    'midl_name_both_missing', 'phone_num_both_missing'
]

# ========== METADATA (for tracking, not training) ==========
metadata_cols = ['id1', 'id2', 'label', 'participation']

print(f"Siamese features: {len(siamese_features)}")
print(f"Baseline model features: {len(model_features)}")

In [ ]:
print(pairs.columns)

In [ ]:
pairs.describe()


Testing out different train test splits. 
1. Entity-Disjoint Split - splitting by "blocking key" or "entity clusters" to ensure train/test independence. Commonly used in entity linkage/duplicate detection. 

2. Stratified Split

1. Can our model identify duplicates among completely new people who happen to fall into the same blocking buckets?"


In [ ]:
from sklearn.model_selection import train_test_split

# Get list of unique person IDs for splitting from ncvoters 
unique_ids = df_ncvoters['id'].unique()  

# Split persons into train (70%), validation (10%), test (20%) - we can adjust this 
train_ids, test_ids = train_test_split(unique_ids, test_size=0.2, random_state=42)
train_ids, val_ids = train_test_split(train_ids, test_size=0.125, random_state=42)  # 0.125 of 80% = 10% of total


print(f"Train IDs: {len(train_ids)}")
print(f"Val IDs: {len(val_ids)}")
print(f"Test IDs: {len(test_ids)}")


# Filter pairs based on id splits 
train_pairs = pairs[(pairs['id_1'].isin(train_ids)) & (pairs['id_2'].isin(train_ids))]
val_pairs = pairs[(pairs['id_1'].isin(val_ids)) & (pairs['id_2'].isin(val_ids))]
test_pairs = pairs[(pairs['id_1'].isin(test_ids)) & (pairs['id_2'].isin(test_ids))]

print(f"Train pairs: {len(train_pairs)}")
print(f"Val pairs: {len(val_pairs)}")
print(f"Test pairs: {len(test_pairs)}")

# X_train = train_pairs[siamese_features]
# y_train = train_pairs['label']


In [ ]:
# Check no ID overlap between sets
train_set = set(train_ids)
val_set = set(val_ids)
test_set = set(test_ids)

assert len(train_set & val_set) == 0, "Train and Val overlap!"
assert len(train_set & test_set) == 0, "Train and Test overlap!"
assert len(val_set & test_set) == 0, "Val and Test overlap!"

print("√ ID sets are disjoint")

# Check no person appears across pair sets
train_all_ids = set(train_pairs['id_1']).union(set(train_pairs['id_2']))
val_all_ids = set(val_pairs['id_1']).union(set(val_pairs['id_2']))
test_all_ids = set(test_pairs['id_1']).union(set(test_pairs['id_2']))

assert len(train_all_ids & val_all_ids) == 0, "ERROR: People appear in train and val!"
assert len(train_all_ids & test_all_ids) == 0, "ERROR: People appear in train and test!"
assert len(val_all_ids & test_all_ids) == 0, "ERROR: People appear in val and test!"

print("√ No person appears in multiple splits")

# Check label distribution
print("\nLabel distribution:")
print(f"Train: {train_pairs['label'].value_counts(normalize=True)}")
print(f"Val: {val_pairs['label'].value_counts(normalize=True)}")
print(f"Test: {test_pairs['label'].value_counts(normalize=True)}")

2. Stratification split - in this case we are asking "can our model classify pairs correctly, even if it's seen these people before in different pairs?"

In [ ]:
#Splitting off test
train_val_df, test_df = train_test_split(
    pairs,
    test_size=0.2,
    stratify=pairs["label"],
    random_state=42
)

# Splittting train and val 
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.125,   # 10% of original data
    stratify=train_val_df["label"],
    random_state=42
)

In [ ]:
#Size 
print("Train size:", len(train_df))
print("Validation size:", len(val_df))
print("Test size:", len(test_df))

In [ ]:
# # Class Distribution 

# # count
# print("\nTrain label counts:")
# print(train_df["label"].value_counts())

# print("\nValidation label counts:")
# print(val_df["label"].value_counts())

# print("\nTest label counts:")
# print(test_df["label"].value_counts())

# # Percentage
# print("\nTrain label %:")
# print(train_df["label"].value_counts(normalize=True) * 100)

# print("\nValidation label %:")
# print(val_df["label"].value_counts(normalize=True) * 100)

# print("\nTest label %:")
# print(test_df["label"].value_counts(normalize=True) * 100)

In [ ]:
# Class distribution 
# Count
summary = pd.DataFrame({
    "Train": train_df["label"].value_counts(),
    "Validation": val_df["label"].value_counts(),
    "Test": test_df["label"].value_counts()
}).fillna(0).astype(int)

summary.loc["Total"] = summary.sum()

print(summary)

# Percentage
summary = pd.DataFrame({
    "Train(%)": train_df["label"].value_counts(normalize=True)*100,
    "Validation(%)": val_df["label"].value_counts(normalize=True)*100,
    "Test(%)": test_df["label"].value_counts(normalize=True)*100
}).fillna(0).astype(int)

summary.loc["Total"] = summary.sum()

print(summary)

 # EDA to determine the best embedding for our model

In [ ]:
import pandas as pd
import numpy as np
from fuzzywuzzy import fuzz
from Levenshtein import distance as levenshtein_distance

# Sample from your pairs dataset
sample_pairs = pairs.sample(min(1000, len(pairs)), random_state=42)

# ========== CHARACTER-LEVEL NOISE ANALYSIS ==========

def analyze_character_noise(df):
    """Detect typos, abbreviations, small edits"""
    
    results = {
        'field': [],
        'avg_edit_distance': [],
        'avg_char_similarity': [],
        'small_edits_pct': []  # Edit distance <= 2
    }
    
    text_fields = ['first_name', 'last_name', 'street_name', 'res_city_desc']
    
    for field in text_fields:
        col1 = f'{field}_1'
        col2 = f'{field}_2'
        
        # Only compare non-empty pairs
        valid = df[(df[col1].notna()) & (df[col2].notna()) & 
                   (df[col1] != '') & (df[col2] != '')]
        
        if len(valid) == 0:
            continue
            
        # Edit distance
        edit_dists = valid.apply(
            lambda x: levenshtein_distance(str(x[col1]), str(x[col2])), 
            axis=1
        )
        
        # Character-level similarity (0-100)
        char_sims = valid.apply(
            lambda x: fuzz.ratio(str(x[col1]), str(x[col2])), 
            axis=1
        )
        
        # Small edits (typos, 1-2 character difference)
        small_edits = (edit_dists <= 2).sum() / len(valid) * 100
        
        results['field'].append(field)
        results['avg_edit_distance'].append(edit_dists.mean())
        results['avg_char_similarity'].append(char_sims.mean())
        results['small_edits_pct'].append(small_edits)
    
    return pd.DataFrame(results)

# ========== SEMANTIC VARIATION ANALYSIS ==========

def analyze_semantic_variation(df):
    """Detect completely different values (semantic differences)"""
    
    results = {
        'field': [],
        'exact_match_pct': [],
        'partial_match_pct': [],  # Share some words
        'completely_different_pct': []
    }
    
    text_fields = ['first_name', 'last_name', 'street_name', 'res_city_desc']
    
    for field in text_fields:
        col1 = f'{field}_1'
        col2 = f'{field}_2'
        
        valid = df[(df[col1].notna()) & (df[col2].notna()) & 
                   (df[col1] != '') & (df[col2] != '')]
        
        if len(valid) == 0:
            continue
        
        # Exact matches
        exact = (valid[col1] == valid[col2]).sum() / len(valid) * 100
        
        # Partial matches (share at least one word)
        def shares_words(row):
            words1 = set(str(row[col1]).lower().split())
            words2 = set(str(row[col2]).lower().split())
            return len(words1 & words2) > 0
        
        partial = valid.apply(shares_words, axis=1).sum() / len(valid) * 100
        
        # Completely different (no shared words)
        completely_diff = 100 - partial
        
        results['field'].append(field)
        results['exact_match_pct'].append(exact)
        results['partial_match_pct'].append(partial - exact)  # Exclude exact
        results['completely_different_pct'].append(completely_diff)
    
    return pd.DataFrame(results)

# ========== RUN ANALYSIS ==========

print("=" * 60)
print("CHARACTER-LEVEL NOISE ANALYSIS")
print("=" * 60)
char_analysis = analyze_character_noise(sample_pairs)
print(char_analysis.to_string(index=False))
print("\nInterpretation:")
print("- Avg edit distance < 3: Mostly typos/abbreviations")
print("- Avg char similarity > 80: High character overlap")
print("- Small edits > 20%: Significant character-level noise")

print("\n" + "=" * 60)
print("SEMANTIC VARIATION ANALYSIS")
print("=" * 60)
semantic_analysis = analyze_semantic_variation(sample_pairs)
print(semantic_analysis.to_string(index=False))
print("\nInterpretation:")
print("- Exact match > 50%: Low variation")
print("- Completely different > 30%: High semantic variation")
print("- Need to distinguish different people, not just match typos")

# ========== SPLIT BY LABEL ==========

print("\n" + "=" * 60)
print("DUPLICATES (label=1) - Should have CHARACTER noise")
print("=" * 60)
duplicates = sample_pairs[sample_pairs['label'] == 1]
if len(duplicates) > 0:
    dup_char = analyze_character_noise(duplicates)
    print(dup_char.to_string(index=False))

print("\n" + "=" * 60)
print("NON-DUPLICATES (label=0) - Should have SEMANTIC differences")
print("=" * 60)
non_duplicates = sample_pairs[sample_pairs['label'] == 0]
if len(non_duplicates) > 0:
    non_dup_semantic = analyze_semantic_variation(non_duplicates)
    print(non_dup_semantic.to_string(index=False))

# ========== VERDICT ==========

print("\n" + "=" * 60)
print("VERDICT: What does our data need?")
print("=" * 60)

# Heuristic decision
avg_edit_dist = char_analysis['avg_edit_distance'].mean()
avg_char_sim = char_analysis['avg_char_similarity'].mean()
completely_diff_pct = semantic_analysis['completely_different_pct'].mean()

if avg_edit_dist < 3 and avg_char_sim > 80:
    print("✓ PRIMARY CHALLENGE: Character-level noise (typos, abbreviations)")
    print("  → RECOMMENDATION: Character CNN or strong character features")
elif completely_diff_pct > 30:
    print("✓ PRIMARY CHALLENGE: Semantic variation (different values)")
    print("  → RECOMMENDATION: SBERT for semantic understanding")
else:
    print("✓ PRIMARY CHALLENGE: BOTH character noise AND semantic variation")
    print("  → RECOMMENDATION: Hybrid (SBERT + character features)")

In [ ]:
# Get all duplicates from your pairs dataset
sample_pairs_duplicates = pairs[pairs['label'] == 1].sample(
    min(1000, len(pairs[pairs['label'] == 1])), 
    random_state=42
)

print(f"Total duplicates in dataset: {len(pairs[pairs['label'] == 1])}")
print(f"Sampled duplicates for analysis: {len(sample_pairs_duplicates)}")

# Run the analysis on duplicates only
print("\n" + "=" * 60)
print("CHARACTER-LEVEL NOISE ANALYSIS (DUPLICATES ONLY)")
print("=" * 60)
char_analysis_dups = analyze_character_noise(sample_pairs_duplicates)
print(char_analysis_dups.to_string(index=False))

print("\n" + "=" * 60)
print("SEMANTIC VARIATION ANALYSIS (DUPLICATES ONLY)")
print("=" * 60)
semantic_analysis_dups = analyze_semantic_variation(sample_pairs_duplicates)
print(semantic_analysis_dups.to_string(index=False))

In [ ]:
#Measure Similarity Distribution for Negatives

from rapidfuzz.fuzz import ratio
import numpy as np

# Sample negatives (label == 0)
neg_sample = pairs[pairs['label'] == 0].sample(
    min(2000, len(pairs[pairs['label'] == 0])),
    random_state=42
)

def compute_name_similarity(row):
    fn_sim = ratio(str(row['first_name_1']), str(row['first_name_2']))
    ln_sim = ratio(str(row['last_name_1']), str(row['last_name_2']))
    return fn_sim, ln_sim, (fn_sim + ln_sim) / 2

sims = neg_sample.apply(compute_name_similarity, axis=1)

neg_sample['fn_sim'] = [s[0] for s in sims]
neg_sample['ln_sim'] = [s[1] for s in sims]
neg_sample['name_sim_avg'] = [s[2] for s in sims]

print("NEGATIVE PAIRS SIMILARITY SUMMARY")
print("="*50)
print(neg_sample[['fn_sim','ln_sim','name_sim_avg']].describe())

# % of hard negatives based on threshold
hard_threshold = 85

hard_pct = (neg_sample['name_sim_avg'] >= hard_threshold).mean() * 100

print(f"\n% of negatives with avg name similarity >= {hard_threshold}: {hard_pct:.2f}%")

In [ ]:
import pandas as pd
import numpy as np
from fuzzywuzzy import fuzz

# ========== HARD NEGATIVE DETECTION ==========

def detect_hard_negatives(df, label=0, sample_size=1000):
    """
    Hard negatives = non-duplicates that LOOK similar (high risk of false positive)
    Easy negatives = obviously different
    """
    
    non_dups = df[df['label'] == label].sample(
        min(sample_size, len(df[df['label'] == label])), 
        random_state=42
    )
    
    results = []
    
   # ['ncid', 'name_sufx_cd', 'first_name', 'midl_name', 'last_name', 'name_sufx_cd','age', 'sex', 'street_name', 'res_city_desc','street_dir', 'street_sufx_cd']
    for idx, row in non_dups.iterrows():
        # Calculate similarity scores
        first_name_sim = fuzz.ratio(str(row['first_name_1']), str(row['first_name_2']))
        last_name_sim = fuzz.ratio(str(row['last_name_1']), str(row['last_name_2']))
        zip_match = 1 if row['zip_code_1'] == row['zip_code_2'] else 0
        city_sim = fuzz.ratio(str(row['res_city_desc_1']), str(row['res_city_desc_2']))
        
        # Calculate overall similarity
        avg_similarity = np.mean([first_name_sim, last_name_sim, city_sim])
        
        results.append({
            'id_1': row['id_1'],
            'id_2': row['id_2'],
            'first_name_sim': first_name_sim,
            'last_name_sim': last_name_sim,
            'zip_match': zip_match,
            'city_sim': city_sim,
            'avg_similarity': avg_similarity,
            'last_name_1': row['last_name_1'],
            'last_name_2': row['last_name_2'],
            'zip_1': row['zip_code_1'],
            'zip_2': row['zip_code_2']
        })
    
    return pd.DataFrame(results)

# Run analysis
print("=" * 80)
print("HARD NEGATIVE ANALYSIS (label=0, non-duplicates)")
print("=" * 80)

hard_neg_analysis = detect_hard_negatives(pairs, label=0, sample_size=1000)

# Categorize by difficulty
hard_neg_analysis['difficulty'] = pd.cut(
    hard_neg_analysis['avg_similarity'],
    bins=[0, 30, 60, 100],
    labels=['Easy (very different)', 'Medium', 'Hard (very similar)']
)

print("\n" + "=" * 80)
print("DISTRIBUTION OF NON-DUPLICATE DIFFICULTY")
print("=" * 80)
print(hard_neg_analysis['difficulty'].value_counts())
print(f"\nPercentage breakdown:")
print(hard_neg_analysis['difficulty'].value_counts(normalize=True) * 100)

# Check for blocking indicators (same last name OR same zip)
print("\n" + "=" * 80)
print("BLOCKING INDICATORS (Signs of hard negatives)")
print("=" * 80)

same_last_name = (hard_neg_analysis['last_name_sim'] > 90).sum()
same_zip = hard_neg_analysis['zip_match'].sum()
both = ((hard_neg_analysis['last_name_sim'] > 90) & 
        (hard_neg_analysis['zip_match'] == 1)).sum()

total = len(hard_neg_analysis)

print(f"Same last name (>90% similar): {same_last_name}/{total} ({same_last_name/total*100:.1f}%)")
print(f"Same zip code: {same_zip}/{total} ({same_zip/total*100:.1f}%)")
print(f"Both (last name + zip): {both}/{total} ({both/total*100:.1f}%)")

# Show hardest negatives
print("\n" + "=" * 80)
print("TOP 10 HARDEST NEGATIVES (most similar, but NOT duplicates)")
print("=" * 80)
hardest = hard_neg_analysis.nlargest(10, 'avg_similarity')[
    ['last_name_1', 'last_name_2', 'zip_1', 'zip_2', 
     'last_name_sim', 'zip_match', 'avg_similarity']
]
print(hardest.to_string(index=False))

# Show easiest negatives
print("\n" + "=" * 80)
print("TOP 10 EASIEST NEGATIVES (very different)")
print("=" * 80)
easiest = hard_neg_analysis.nsmallest(10, 'avg_similarity')[
    ['last_name_1', 'last_name_2', 'zip_1', 'zip_2', 
     'last_name_sim', 'zip_match', 'avg_similarity']
]
print(easiest.to_string(index=False))

# ========== VERDICT ==========
print("\n" + "=" * 80)
print("VERDICT: Do you have hard negatives?")
print("=" * 80)

hard_pct = (hard_neg_analysis['difficulty'] == 'Hard (very similar)').sum() / total * 100
blocking_pct = max(same_last_name/total*100, same_zip/total*100)

if blocking_pct > 50:
    print(f"✓ YES - You have HARD NEGATIVES")
    print(f"  - {blocking_pct:.1f}% share blocking keys (last name or zip)")
    print(f"  - These are similar records that are NOT duplicates")
    print(f"  - Model must learn fine-grained distinctions")
    print(f"\n  → This is GOOD for training (challenging, realistic)")
elif hard_pct > 20:
    print(f"✓ SOME - You have MODERATE hard negatives")
    print(f"  - {hard_pct:.1f}% are quite similar but not duplicates")
else:
    print(f" NO - Mostly EASY negatives")
    print(f"  - {hard_pct:.1f}% are similar")
    print(f"  - Non-duplicates are obviously different from the textbook definition")
    print(f"  - Dataset might be too easy or we haven't found the correct pattern in the data")

In [ ]:
# ========== VERIFY BLOCKING STRATEGY ==========

def check_blocking_coverage(df, label=0):
    """Check what % of non-duplicates share ANY field"""
    
    non_dups = df[df['label'] == label]
    
    print("=" * 80)
    print(f"BLOCKING VERIFICATION (n={len(non_dups)} non-duplicate pairs)")
    print("=" * 80)
    
    # Check each potential blocking key
    blocking_stats = {}
    
    # Exact matches
    blocking_stats['last_name'] = (non_dups['last_name_1'] == non_dups['last_name_2']).sum()
    blocking_stats['first_name'] = (non_dups['first_name_1'] == non_dups['first_name_2']).sum()
    blocking_stats['zip_code'] = (non_dups['zip_code_1'] == non_dups['zip_code_2']).sum()
    blocking_stats['city'] = (non_dups['res_city_desc_1'] == non_dups['res_city_desc_2']).sum()
    blocking_stats['phone'] = (non_dups['phone_num_1'] == non_dups['phone_num_2']).sum()
    
    # Partial matches (first 3 chars for fuzzy blocking)
    non_dups_clean = non_dups.copy()
    non_dups_clean['last_name_1_prefix'] = non_dups_clean['last_name_1'].astype(str).str[:3]
    non_dups_clean['last_name_2_prefix'] = non_dups_clean['last_name_2'].astype(str).str[:3]
    blocking_stats['last_name_prefix3'] = (non_dups_clean['last_name_1_prefix'] == 
                                           non_dups_clean['last_name_2_prefix']).sum()
    
    # Print results
    print("\nBlocking Key Match Rates:")
    print("-" * 80)
    for key, count in blocking_stats.items():
        pct = count / len(non_dups) * 100
        print(f"{key:25s}: {count:6d} / {len(non_dups)} ({pct:5.1f}%)")
    
    # Check if ANY blocking key matches
    any_match = (
        (non_dups['last_name_1'] == non_dups['last_name_2']) |
        (non_dups['zip_code_1'] == non_dups['zip_code_2']) |
        (non_dups['res_city_desc_1'] == non_dups['res_city_desc_2']) |
        (non_dups['first_name_1'] == non_dups['first_name_2'])
    ).sum()
    
    print("-" * 80)
    print(f"ANY field matches         : {any_match:6d} / {len(non_dups)} ({any_match/len(non_dups)*100:5.1f}%)")
    print("-" * 80)
    
    # Verdict
    if any_match / len(non_dups) > 0.5:
        print("\n✓ YES - Blocking was applied (>50% share at least one field)")
    elif any_match / len(non_dups) > 0.2:
        print("\n? MAYBE - Some blocking (20-50% share fields)")
    else:
        print("\n✗ NO - Blocking was NOT applied (<20% share fields)")
        print("  → These appear to be random non-duplicate pairs")

# Run the check
check_blocking_coverage(pairs, label=0)

# ========== COMPARE TO DUPLICATES ==========

print("\n" + "=" * 80)
print("COMPARISON: Duplicates vs Non-Duplicates")
print("=" * 80)

duplicates = pairs[pairs['label'] == 1]
non_duplicates = pairs[pairs['label'] == 0]

comparison = pd.DataFrame({
    'Blocking Key': ['Last Name Match', 'Zip Match', 'City Match', 'First Name Match'],
    'Duplicates (%)': [
        (duplicates['last_name_1'] == duplicates['last_name_2']).sum() / len(duplicates) * 100,
        (duplicates['zip_code_1'] == duplicates['zip_code_2']).sum() / len(duplicates) * 100,
        (duplicates['res_city_desc_1'] == duplicates['res_city_desc_2']).sum() / len(duplicates) * 100,
        (duplicates['first_name_1'] == duplicates['first_name_2']).sum() / len(duplicates) * 100
    ],
    'Non-Duplicates (%)': [
        (non_duplicates['last_name_1'] == non_duplicates['last_name_2']).sum() / len(non_duplicates) * 100,
        (non_duplicates['zip_code_1'] == non_duplicates['zip_code_2']).sum() / len(non_duplicates) * 100,
        (non_duplicates['res_city_desc_1'] == non_duplicates['res_city_desc_2']).sum() / len(non_duplicates) * 100,
        (non_duplicates['first_name_1'] == non_duplicates['first_name_2']).sum() / len(non_duplicates) * 100
    ]
})

print(comparison.to_string(index=False))

print("\n" + "=" * 80)
print("INTERPRETATION:")
print("=" * 80)
print("If duplicates have HIGH match rates and non-duplicates have LOW match rates:")
print("  → Dataset is EASY (duplicates obvious, non-duplicates obvious)")
print("\nIf BOTH have similar match rates:")
print("  → Dataset is HARD (need fine-grained discrimination)")

In [ ]:
# Fix: Check for non-empty phones BEFORE matching
non_dups_real_phone_match = pairs[
    (pairs['label'] == 0) & 
    (pairs['phone_num_1'] == pairs['phone_num_2']) &
    (pairs['phone_num_1'].notna()) &
    (pairs['phone_num_1'] != '') &
    (pairs['phone_num_1'].str.len() > 0)  # Extra safety
]

print(f"Non-duplicates with REAL matching phone: {len(non_dups_real_phone_match)}")

# Recalculate blocking stats properly
def check_blocking_corrected(df, label=0):
    """Fixed version: exclude empty strings"""
    
    non_dups = df[df['label'] == 0]
    
    print("=" * 80)
    print("CORRECTED BLOCKING VERIFICATION (excluding empty strings)")
    print("=" * 80)
    
    # Helper function to check non-empty matches
    def non_empty_match(col1, col2):
        return (
            (non_dups[col1] == non_dups[col2]) &
            (non_dups[col1].notna()) &
            (non_dups[col1] != '') &
            (non_dups[col1].str.len() > 0)
        ).sum()
    
    blocking_stats = {
        'last_name': non_empty_match('last_name_1', 'last_name_2'),
        'first_name': non_empty_match('first_name_1', 'first_name_2'),
        'zip_code': non_empty_match('zip_code_1', 'zip_code_2'),
        'city': non_empty_match('res_city_desc_1', 'res_city_desc_2'),
        'phone': non_empty_match('phone_num_1', 'phone_num_2'),
    }
    
    print("\nBlocking Key Match Rates (non-empty only):")
    print("-" * 80)
    for key, count in blocking_stats.items():
        pct = count / len(non_dups) * 100
        print(f"{key:25s}: {count:6d} / {len(non_dups)} ({pct:5.1f}%)")
    
    return blocking_stats

check_blocking_corrected(pairs, label=0)

In [ ]:
# Investigate the phone matches more deeply
print("=" * 80)
print("PHONE NUMBER INVESTIGATION")
print("=" * 80)

# What are the actual phone values that match?
phone_value_counts = non_dups_real_phone_match['phone_num_1'].value_counts()

print(f"\nTotal non-duplicates with matching phones: {len(non_dups_real_phone_match)}")
print(f"Unique phone values that match: {len(phone_value_counts)}")
print(f"\nTop 20 most common matching phone values:")
print(phone_value_counts.head(20))

# Check a specific example
if len(phone_value_counts) > 0:
    most_common_phone = phone_value_counts.index[0]
    print(f"\n" + "=" * 80)
    print(f"Example: Phone '{most_common_phone}' appears in {phone_value_counts.iloc[0]} non-duplicate pairs")
    print("=" * 80)
    
    sample = non_dups_real_phone_match[non_dups_real_phone_match['phone_num_1'] == most_common_phone].head(5)
    print(sample[['id_1', 'id_2', 'phone_num_1', 'first_name_1', 'first_name_2', 
                  'last_name_1', 'last_name_2', 'zip_code_1', 'zip_code_2']])

# Check if phone is actually a placeholder value
print("\n" + "=" * 80)
print("Are these placeholder/default phone numbers?")
print("=" * 80)

# Common placeholders
placeholders = ['0000000000', '9999999999', '1111111111', 'UNKNOWN', 'NA', 'NONE']
placeholder_count = non_dups_real_phone_match[
    non_dups_real_phone_match['phone_num_1'].isin(placeholders)
].shape[0]

print(f"Pairs with obvious placeholder phones: {placeholder_count} / {len(non_dups_real_phone_match)}")

In [ ]:
pairs.describe()

In [ ]:
# Deep dive into phone field
print("=" * 80)
print("WHAT IS 'EMPTY' IN PHONE FIELD?")
print("=" * 80)

# Check unique values in phone_num_1
unique_phones = pairs['phone_num_1'].unique()
print(f"Unique values in phone_num_1: {len(unique_phones)}")
print(f"\nFirst 20 unique values (with repr to see hidden chars):")
for i, val in enumerate(unique_phones[:20]):
    print(f"{i}: {repr(val)} | Length: {len(str(val)) if pd.notna(val) else 'NaN'}")

# Check the 28,121 matching pairs specifically
print("\n" + "=" * 80)
print("What value is in the 28,121 matching phone pairs?")
print("=" * 80)

matching_phone_val = non_dups_real_phone_match['phone_num_1'].iloc[0] if len(non_dups_real_phone_match) > 0 else None
print(f"First matching phone value: {repr(matching_phone_val)}")
print(f"Type: {type(matching_phone_val)}")
print(f"Length: {len(str(matching_phone_val)) if pd.notna(matching_phone_val) else 'NaN'}")

# Count how many pairs have this exact value
if matching_phone_val is not None:
    same_val_count = ((pairs['phone_num_1'] == matching_phone_val) & 
                      (pairs['phone_num_2'] == matching_phone_val)).sum()
    print(f"Total pairs with this phone value: {same_val_count}")

In [ ]:
# FINAL CORRECTED VERSION: Exclude single space
def check_blocking_final(df, label=0):
    """Properly exclude empty strings AND single spaces"""
    
    non_dups = df[df['label'] == 0]
    
    print("=" * 80)
    print("FINAL CORRECTED BLOCKING VERIFICATION")
    print("=" * 80)
    
    def non_empty_match(col1, col2):
        return (
            (non_dups[col1] == non_dups[col2]) &
            (non_dups[col1].notna()) &
            (non_dups[col1] != '') &
            (non_dups[col1] != ' ') &  # Exclude single space
            (non_dups[col1].str.strip() != '')  # Exclude whitespace
        ).sum()
    
    blocking_stats = {
        'last_name': non_empty_match('last_name_1', 'last_name_2'),
        'first_name': non_empty_match('first_name_1', 'first_name_2'),
        'zip_code': non_empty_match('zip_code_1', 'zip_code_2'),
        'city': non_empty_match('res_city_desc_1', 'res_city_desc_2'),
        'phone': non_empty_match('phone_num_1', 'phone_num_2'),
    }
    
    print("\nBlocking Key Match Rates (excluding empty/whitespace):")
    print("-" * 80)
    for key, count in blocking_stats.items():
        pct = count / len(non_dups) * 100
        print(f"{key:25s}: {count:6d} / {len(non_dups)} ({pct:5.1f}%)")
    
    # ANY field matches
    any_match = (
        (non_empty_match('last_name_1', 'last_name_2') > 0) |
        (non_empty_match('zip_code_1', 'zip_code_2') > 0) |
        (non_empty_match('city_1', 'city_2') > 0)
    )
    
    print("-" * 80)
    total_any = sum(blocking_stats.values())
    print(f"ANY field matches         : ~{total_any:6d} / {len(non_dups)} (~{total_any/len(non_dups)*100:5.1f}%)")
    
    return blocking_stats

check_blocking_final(pairs, label=0)


Names: Pure Character-Level Noise

First name: 0.07 avg edit distance, 99% char similarity, 98.9% are 1-2 char typos!
Last name: 0.9 avg edit distance, 89% char similarity, 85.8% are small edits
97.6% exact first name match, 85.7% exact last name match

Translation: Duplicates are almost always the SAME person with minor typos/variations. 

98.9% of duplicates differ by 1-2 characters in name
Addresses are noise (person moved)

In [ ]:
# Analyze ONLY duplicates (label=1)
duplicates_only = pairs[pairs['label'] == 1]

print("=" * 80)
print(f"DUPLICATE PAIRS ONLY (label=1) - n={len(duplicates_only)}")
print("=" * 80)

# Character-level analysis on duplicates
char_dup = analyze_character_noise(duplicates_only.sample(min(1000, len(duplicates_only))))
print("\nCHARACTER-LEVEL NOISE IN DUPLICATES:")
print(char_dup.to_string(index=False))

# Semantic analysis on duplicates  
semantic_dup = analyze_semantic_variation(duplicates_only.sample(min(1000, len(duplicates_only))))
print("\nSEMANTIC VARIATION IN DUPLICATES:")
print(semantic_dup.to_string(index=False))

* First Names (2.1% differ):

Mostly character-level typos: "semaje" vs "samaje" (1-2 edits)
Spacing errors: "de ontaye" vs "deontaye", "ky wana" vs "kywana"
Typos: "maruice" vs "maurice", "joesph" vs "joseph"
A few completely different: "reginald" vs "todd" (7 edits) - probably data errors or middle name confusion


* Last Names (14.4% differ):

Some typos: "christensen" vs "christersen" (1 edit), "millard" vs "hillard" (1 edit)
BUT MOSTLY: Completely different names - "trotter" vs "mitchell" (7 edits), "washington" vs "holland" (9 edits)


* We have small character level duplicates


* ~2% first names have typos
* ~2-3% last names have typos
* The last name pattern we are seeing are name changes

* 14% different last names = marriage/divorce

* "washington" vs "holland" (maiden name vs married name)
* "smith" vs "townsend"

Our pattern is actually a bit of character
And then actual legal name changes
Character IDENTITY CHANGES OVER TIME

* We want to LEARN: "Same first name + different last name + different address = still same person (probably married/divorced)"
our data has:
98% exact name matches (easy, no CNN needed)
2% typos (CNN would help marginally)
14% name changes (neither CNN nor SBERT will help)





A little discouraged by these patterns... this data is not matching text book defintion. 

In [ ]:
# Find duplicates where names DON'T exactly match
duplicates = pairs[pairs['label'] == 1]

# Cases where first name differs
first_name_differs = duplicates[duplicates['first_name_1'] != duplicates['first_name_2']]
print(f"Duplicates with different first names: {len(first_name_differs)} / {len(duplicates)} ({len(first_name_differs)/len(duplicates)*100:.1f}%)")

# Cases where last name differs  
last_name_differs = duplicates[duplicates['last_name_1'] != duplicates['last_name_2']]
print(f"Duplicates with different last names: {len(last_name_differs)} / {len(duplicates)} ({len(last_name_differs)/len(duplicates)*100:.1f}%)")

print("\n" + "=" * 80)
print("EXAMPLES WHERE FIRST NAME DIFFERS (Character-level typos?)")
print("=" * 80)

# Show ALL first name differences (or first 50 if too many)
for idx, row in first_name_differs.head(50).iterrows():
    name1 = row['first_name_1']
    name2 = row['first_name_2']
    edit_dist = levenshtein_distance(name1, name2)
    print(f"{name1:20s} vs {name2:20s} | Edit distance: {edit_dist}")

print("\n" + "=" * 80)
print("EXAMPLES WHERE LAST NAME DIFFERS")
print("=" * 80)

for idx, row in last_name_differs.head(50).iterrows():
    name1 = row['last_name_1']
    name2 = row['last_name_2']
    edit_dist = levenshtein_distance(name1, name2)
    print(f"{name1:20s} vs {name2:20s} | Edit distance: {edit_dist}")

* 100% of duplicates share NCID 0% share Voter ID. NCID can be used as a proxy for SSN:


* √ ncid is the true person identifier (like SSN)- ground truth - not just label 0/1
* √ Duplicates are same person at different points in time
* √ You're learning: "How does one person's voter registration evolve?"


In [ ]:
# Check if duplicates share ncid or voter_reg_num
duplicates = pairs[pairs['label'] == 1]

# Do duplicates share ncid?
ncid_match = (duplicates['ncid_1'] == duplicates['ncid_2']).sum()
ncid_both_present = ((duplicates['ncid_1'].notna()) & (duplicates['ncid_2'].notna())).sum()

print(f"Duplicates with matching ncid: {ncid_match} / {ncid_both_present} ({ncid_match/ncid_both_present*100:.1f}%)")

# Do duplicates share voter_reg_num?
voter_match = (duplicates['voter_reg_num_1'] == duplicates['voter_reg_num_2']).sum()
voter_both_present = ((duplicates['voter_reg_num_1'].notna()) & (duplicates['voter_reg_num_2'].notna())).sum()

print(f"Duplicates with matching voter_reg_num: {voter_match} / {voter_both_present} ({voter_match/voter_both_present*100:.1f}%)")

# Show examples where ncid matches but names differ
print("\n" + "=" * 80)
print("EXAMPLES: Same ncid, different names (name changes over time)")
print("=" * 80)

name_differs_ncid_same = duplicates[
    (duplicates['ncid_1'] == duplicates['ncid_2']) &
    (duplicates['last_name_1'] != duplicates['last_name_2'])
].head(10)

for idx, row in name_differs_ncid_same.iterrows():
    print(f"ncid: {row['ncid_1']}")
    print(f"  Record 1: {row['first_name_1']} {row['last_name_1']}")
    print(f"  Record 2: {row['first_name_2']} {row['last_name_2']}")
    print()

Our model needs to learn: 
* Pattern 1: Marriage/Divorce (14% of duplicates)
* Pattern 2: Address Changes (78% different cities)
* Pattern 3: Data Entry Typos (2% of names)

Q: We want to predict duplicates WITHOUT using ncid (simulate real-world where we don't have SSN)

# Non duplicate analysis

In [ ]:
# ========== REALISTIC HARD NEGATIVES ==========

print("=" * 80)
print("REALISTIC HARD NEGATIVES: Natural Confusion Cases")
print("=" * 80)

non_dups = non_duplicates.sample(min(10000, len(non_duplicates)), random_state=42)

# ========== 1. FAMILY MEMBERS (Same last name + zip) ==========

print("\n" + "=" * 80)
print("CATEGORY 1: Likely Family Members (same last name + zip)")
print("=" * 80)

family_members = non_dups[
    (non_dups['last_name_1'] == non_dups['last_name_2']) &
    (non_dups['zip_code_1'] == non_dups['zip_code_2']) &
    (non_dups['first_name_1'] != non_dups['first_name_2'])
]

print(f"Found {len(family_members)} potential family member pairs")
print(f"Percentage: {len(family_members)/len(non_dups)*100:.2f}%\n")

# Show examples
for idx, row in family_members.head(15).iterrows():
    first_sim = fuzz.ratio(row['first_name_1'], row['first_name_2'])
    print(f"{row['first_name_1']:15s} {row['last_name_1']:15s} (zip: {row['zip_code_1']})")
    print(f"{row['first_name_2']:15s} {row['last_name_2']:15s} (zip: {row['zip_code_2']})")
    print(f"  First name similarity: {first_sim}% | Age: {row.get('age_1', 'N/A')} vs {row.get('age_2', 'N/A')}")
    print()

# ========== 2. SIMILAR FIRST NAMES (Common name variants) ==========

print("=" * 80)
print("CATEGORY 2: Name Variants (Jon/John, Beth/Elizabeth)")
print("=" * 80)

# Find pairs with high first name similarity but different
similar_first = non_dups[
    (non_dups['first_name_1'] != non_dups['first_name_2'])
].copy()

similar_first['first_sim'] = similar_first.apply(
    lambda x: fuzz.ratio(str(x['first_name_1']), str(x['first_name_2'])),
    axis=1
)

name_variants = similar_first[similar_first['first_sim'] > 70].sort_values('first_sim', ascending=False)

print(f"Found {len(name_variants)} pairs with similar first names (>70% similarity)")
print(f"Percentage: {len(name_variants)/len(non_dups)*100:.2f}%\n")

for idx, row in name_variants.head(15).iterrows():
    print(f"{row['first_name_1']:15s} vs {row['first_name_2']:15s} (similarity: {row['first_sim']:.0f}%)")
    print(f"  Last names: {row['last_name_1']} vs {row['last_name_2']}")
    print(f"  Zip: {row['zip_code_1']} vs {row['zip_code_2']}")
    print()

# ========== 3. SAME STREET, DIFFERENT HOUSE NUMBER ==========

print("=" * 80)
print("CATEGORY 3: Neighbors (same street, different house number)")
print("=" * 80)

neighbors = non_dups[
    (non_dups['street_name_1'] == non_dups['street_name_2']) &
    (non_dups['zip_code_1'] == non_dups['zip_code_2']) &
    (non_dups['house_num_1'] != non_dups['house_num_2'])
]

print(f"Found {len(neighbors)} neighbor pairs")
print(f"Percentage: {len(neighbors)/len(non_dups)*100:.2f}%\n")

for idx, row in neighbors.head(15).iterrows():
    print(f"{row['first_name_1']} {row['last_name_1']}: {row['house_num_1']} {row['street_name_1']}")
    print(f"{row['first_name_2']} {row['last_name_2']}: {row['house_num_2']} {row['street_name_2']}")
    print()

# ========== 4. COMMON NAMES IN SAME CITY ==========

print("=" * 80)
print("CATEGORY 4: Common Names (e.g., multiple John Smiths)")
print("=" * 80)

# Find most common name combinations in non-duplicates
non_dups['full_name_1'] = non_dups['first_name_1'] + ' ' + non_dups['last_name_1']
non_dups['full_name_2'] = non_dups['first_name_2'] + ' ' + non_dups['last_name_2']

# Count how often each name appears
from collections import Counter
all_names = list(non_dups['full_name_1']) + list(non_dups['full_name_2'])
name_counts = Counter(all_names)

common_names = {name: count for name, count in name_counts.items() if count > 10}

print(f"Found {len(common_names)} names appearing 10+ times in non-duplicates\n")
print("Most common names (potential confusion):")
for name, count in sorted(common_names.items(), key=lambda x: x[1], reverse=True)[:20]:
    print(f"  {name}: {count} occurrences")

# ========== 5. AGE-BASED PATTERNS ==========

if 'age_1' in non_dups.columns and 'age_2' in non_dups.columns:
    print("\n" + "=" * 80)
    print("CATEGORY 5: Similar Ages (potential siblings/peers)")
    print("=" * 80)
    
    similar_age_family = family_members.copy()
    similar_age_family['age_diff'] = abs(
        pd.to_numeric(similar_age_family['age_1'], errors='coerce') - 
        pd.to_numeric(similar_age_family['age_2'], errors='coerce')
    )
    
    siblings = similar_age_family[similar_age_family['age_diff'] < 10]
    
    print(f"Family members with <10 year age difference: {len(siblings)}")
    print("(Potential siblings/spouses)\n")
    
    for idx, row in siblings.head(10).iterrows():
        print(f"{row['first_name_1']} {row['last_name_1']} (age: {row['age_1']})")
        print(f"{row['first_name_2']} {row['last_name_2']} (age: {row['age_2']})")
        print(f"  Age difference: {row['age_diff']:.0f} years | Zip: {row['zip_code_1']}")
        print()

# ========== SUMMARY ==========

print("=" * 80)
print("REAL HARD NEGATIVES SUMMARY")
print("=" * 80)

categories = {
    'Family members (same last name + zip)': len(family_members),
    'Name variants (similar first names)': len(name_variants),
    'Neighbors (same street)': len(neighbors),
    'Common names (10+ occurrences)': len(common_names)
}

total_hard = sum(categories.values())

print(f"\nOut of {len(non_dups)} non-duplicates sampled:\n")
for category, count in categories.items():
    pct = count / len(non_dups) * 100
    print(f"{category:45s}: {count:5d} ({pct:5.2f}%)")

print(f"\n{'Total realistic hard negatives':45s}: ~{total_hard:5d}")

print("\n" + "=" * 80)
print("WHAT YOUR MODEL MUST LEARN")
print("=" * 80)
print("""
The model must distinguish:
1. Same person with name change vs Family members (same last name)
2. Same person with typo vs Different person with similar name
3. Same person who moved vs Neighbors (same street)
4. Same person over time vs Common name collision (John Smith #1 vs John Smith #2)

These are REALISTIC challenges that require learning subtle patterns!
""")

In [ ]:
# ========== NEGATIVES THAT LOOK LIKE POSITIVES ==========

print("=" * 80)
print("HARD NEGATIVES: Cases that look like duplicates but aren't")
print("=" * 80)

non_dups = non_duplicates.sample(min(10000, len(non_duplicates)), random_state=42)

# ========== 1. SAME FIRST NAME, DIFFERENT LAST NAME ==========
# (Mirrors the 14% marriage/divorce pattern in duplicates)

print("\n" + "=" * 80)
print("PATTERN 1: Same first name + different last name")
print("(Could be marriage, or could be different people)")
print("=" * 80)

same_first_diff_last = non_dups[
    (non_dups['first_name_1'] == non_dups['first_name_2']) &
    (non_dups['last_name_1'] != non_dups['last_name_2'])
]

print(f"Found {len(same_first_diff_last)} cases ({len(same_first_diff_last)/len(non_dups)*100:.2f}%)")
print("\nExamples (model must learn these are DIFFERENT people):\n")

for idx, row in same_first_diff_last.head(20).iterrows():
    print(f"{row['first_name_1']:15s} {row['last_name_1']:15s} (ncid: {row['ncid_1'][:8]})")
    print(f"{row['first_name_2']:15s} {row['last_name_2']:15s} (ncid: {row['ncid_2'][:8]})")
    print(f"  Zip: {row['zip_code_1']} vs {row['zip_code_2']}")
    print()

# ========== 2. SIMILAR FIRST NAME + DIFFERENT LAST NAME ==========
# (Mirrors the 2% typo pattern + 14% name change)

print("=" * 80)
print("PATTERN 2: Similar first name (typo-like) + different last name")
print("(Could be same person with typo, or different people)")
print("=" * 80)

similar_first_diff_last = non_dups[
    (non_dups['first_name_1'] != non_dups['first_name_2']) &
    (non_dups['last_name_1'] != non_dups['last_name_2'])
].copy()

# Calculate first name similarity
similar_first_diff_last['first_sim'] = similar_first_diff_last.apply(
    lambda x: fuzz.ratio(str(x['first_name_1']), str(x['first_name_2'])),
    axis=1
)

# High similarity (looks like typo)
typo_like = similar_first_diff_last[
    similar_first_diff_last['first_sim'] > 80
].sort_values('first_sim', ascending=False)

print(f"Found {len(typo_like)} cases with >80% first name similarity ({len(typo_like)/len(non_dups)*100:.2f}%)")
print("\nExamples (looks like typo + name change, but different people):\n")

for idx, row in typo_like.head(20).iterrows():
    print(f"{row['first_name_1']:15s} {row['last_name_1']:15s}")
    print(f"{row['first_name_2']:15s} {row['last_name_2']:15s}")
    print(f"  First name similarity: {row['first_sim']:.0f}%")
    print(f"  Zip: {row['zip_code_1']} vs {row['zip_code_2']}")
    print()

# ========== 3. SAME FIRST NAME + SAME LAST NAME ==========
# (Should be easy to confuse, but different ncid)

print("=" * 80)
print("PATTERN 3: Same first + last name (different people!)")
print("(Classic John Smith problem)")
print("=" * 80)

same_both_names = non_dups[
    (non_dups['first_name_1'] == non_dups['first_name_2']) &
    (non_dups['last_name_1'] == non_dups['last_name_2'])
]

print(f"Found {len(same_both_names)} cases ({len(same_both_names)/len(non_dups)*100:.2f}%)")
print("\nExamples (identical names, must use address/age to distinguish):\n")

for idx, row in same_both_names.head(20).iterrows():
    print(f"{row['first_name_1']} {row['last_name_1']}")
    print(f"  Person 1: ncid={row['ncid_1'][:8]}, zip={row['zip_code_1']}, age={row.get('age_1', 'N/A')}")
    print(f"  Person 2: ncid={row['ncid_2'][:8]}, zip={row['zip_code_2']}, age={row.get('age_2', 'N/A')}")
    print()

# ========== 4. DIFFERENT ADDRESSES (like duplicates who moved) ==========

print("=" * 80)
print("PATTERN 4: Different cities (like people who moved, but different people)")
print("=" * 80)

diff_cities = non_dups[
    (non_dups['res_city_desc_1'] != non_dups['res_city_desc_2'])
]

print(f"Found {len(diff_cities)} cases with different cities ({len(diff_cities)/len(non_dups)*100:.2f}%)")

# Among these, find ones with same first name
diff_city_same_first = diff_cities[
    diff_cities['first_name_1'] == diff_cities['first_name_2']
]

print(f"  Of these, {len(diff_city_same_first)} have same first name")
print("\nExamples (same first name, different cities - not same person):\n")

for idx, row in diff_city_same_first.head(15).iterrows():
    print(f"{row['first_name_1']} {row['last_name_1']}, {row['res_city_desc_1']}")
    print(f"{row['first_name_2']} {row['last_name_2']}, {row['res_city_desc_2']}")
    print()

# ========== SUMMARY: COMPARE TO DUPLICATE PATTERNS ==========

print("=" * 80)
print("COMPARISON: Duplicates vs Hard Negatives")
print("=" * 80)

duplicates = pairs[pairs['label'] == 1]

comparison = pd.DataFrame({
    'Pattern': [
        'Same first, diff last',
        'Similar first (>80%), diff last',
        'Same first + last',
        'Different cities'
    ],
    'In Duplicates (%)': [
        (duplicates['first_name_1'] == duplicates['first_name_2']).sum() / len(duplicates) * 100,
        0,  # Would need to calculate
        ((duplicates['first_name_1'] == duplicates['first_name_2']) & 
         (duplicates['last_name_1'] == duplicates['last_name_2'])).sum() / len(duplicates) * 100,
        (duplicates['res_city_desc_1'] != duplicates['res_city_desc_2']).sum() / len(duplicates) * 100
    ],
    'In Non-Duplicates (%)': [
        len(same_first_diff_last) / len(non_dups) * 100,
        len(typo_like) / len(non_dups) * 100,
        len(same_both_names) / len(non_dups) * 100,
        len(diff_cities) / len(non_dups) * 100
    ]
})

print(comparison.to_string(index=False))

print("\n" + "=" * 80)
print("KEY INSIGHT")
print("=" * 80)
print("""
If duplicates and non-duplicates have SIMILAR patterns:
→ Task is HARD (model must learn subtle differences)

If patterns are DIFFERENT:
→ Task is EASY (clear separation)

The model must learn:
- "Same first name + diff last" can be EITHER duplicate (marriage) OR different people
- "Different cities" can be EITHER duplicate (moved) OR different people
- Must use OTHER signals (age, middle name, etc.) to decide
""")

We know one of our major differentiators are that the model has to learn same first name with last name changes...  and these are the hard negatives. "If both duplicates AND non-duplicates have 'same first name + different last name', how does the model learn to distinguish them?"

What We Know:
Duplicates (label=1):

97.9% same first name
14% different last name (marriage)
78% different cities (moved)

Non-duplicates (label=0):

1.46% same first name + different last name
99%+ different cities

In [ ]:
NCID defines true entity.
	•	Use NCID for grouping.
	•	Do entity-disjoint split by NCID.
	•	Drop NCID from features.
	•	Hard negatives exist but are rare.
	•	Model must learn beyond name similarity.

⸻
